# LeetCode 3168 — Minimum Number of Chairs in a Waiting Room bằng Python

Link đề: https://leetcode.com/problems/minimum-number-of-chairs-in-a-waiting-room/ · Độ khó: **Easy** · Chủ đề: String, Simulation (mô phỏng)

Skeleton LeetCode:

```python
class Solution:
    def minimumChairs(self, s: str) -> int:
        ...
```

Ghi chú: đề và các ví dụ được viết lại theo đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code, nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: chuỗi `s` mô tả từng giây trong phòng chờ — `'E'` (Enter) là một người **vào** và ngồi một ghế, `'L'` (Leave) là một người **ra** và trả lại một ghế; cần tìm **số ghế tối thiểu** để lúc nào cũng đủ ghế cho mọi người đang ở trong phòng.
2. Nắm các công cụ Python: duyệt chuỗi bằng `for ch in s`, biến đếm `cur`, **`max`** để ghi nhận kỷ lục, `if / else`, `itertools.accumulate`, `dict` ánh xạ ký tự → `+1/-1`, và `list` làm ngăn xếp (`append`, `pop`).
3. Đi từ cách **thử từng số ghế** (`c = 1, 2, 3, ...`, mô phỏng kiểm tra) → **một biến đếm `cur` + `max`** → **tổng cộng dồn `accumulate`** → **mô phỏng thật bằng kho ghế trống** → **công thức `E − L` trên mỗi tiền tố**.
4. Hiểu mấu chốt: số ghế cần thiết **bằng số người đông nhất từng ở trong phòng cùng lúc**, tức **giá trị lớn nhất của (số `E` − số `L`) trên mọi tiền tố** — phải **ghi lại đỉnh** trong lúc chạy, không phải số người lúc cuối.
5. Tránh các lỗi kinh điển: **đếm tổng số `'E'`** (bỏ qua việc người ra trả ghế), **trả về `cur` cuối cùng** thay vì giá trị lớn nhất, **quên cập nhật `max`** sau mỗi bước `E`, và nhầm hướng `+1/-1` giữa `E` và `L`.

## 2. Đề bài & ví dụ

Cho một chuỗi `s`. Mô phỏng các sự kiện tại mỗi giây `i`:

- Nếu `s[i] == 'E'`: một người **vào** phòng chờ và ngồi vào **một** ghế.
- Nếu `s[i] == 'L'`: một người **rời** phòng chờ, giải phóng **một** ghế.

Trả về **số ghế tối thiểu** cần có sao cho mọi người vào phòng đều có ghế, biết rằng ban đầu phòng **trống**.

**Ví dụ 1**

- Input: `s = "EEEEEEE"`
- Output: `7`
- Giải thích: mỗi giây có một người vào và không ai ra, nên cần tối thiểu `7` ghế.

**Ví dụ 2**

- Input: `s = "ELELEEL"`
- Output: `2`
- Giải thích: số người trong phòng sau mỗi giây lần lượt là `1, 0, 1, 0, 1, 2, 1`; lớn nhất là `2` (giây `5`).

**Ví dụ 3**

- Input: `s = "ELEELEELLL"`
- Output: `3`
- Giải thích: số người sau mỗi giây là `1, 0, 1, 2, 1, 2, 3, 2, 1, 0`; lớn nhất là `3`.

**Ràng buộc:** `1 <= s.length <= 50`; `s` chỉ gồm các chữ `'E'` và `'L'`; `s` là một dãy vào/ra **hợp lệ** (không ai rời đi khi phòng đang trống).

In [1]:
from typing import List
from itertools import accumulate
import random
import itertools

examples = [
    ("EEEEEEE", 7),
    ("ELELEEL", 2),
    ("ELEELEELLL", 3),
]
for s, expected in examples:
    print(repr(s), "-> kỳ vọng", expected)

'EEEEEEE' -> kỳ vọng 7
'ELELEEL' -> kỳ vọng 2
'ELEELEELLL' -> kỳ vọng 3


## 3. Kiến thức nền

### 3.1. Duyệt từng ký tự của chuỗi

Chuỗi trong Python là dãy ký tự; `for ch in s` lấy từng ký tự theo thứ tự từ trái sang phải. `enumerate(s)` cho thêm chỉ số (ở bài này chỉ số chính là "giây" `i`). So sánh ký tự bằng `==`: `ch == 'E'`.

In [2]:
s = "ELE"
for ch in s:
    print(ch, ch == "E")
for i, ch in enumerate(s):
    print(f"giây {i}: {ch}")

E True
L False
E True
giây 0: E
giây 1: L
giây 2: E


### 3.2. Biến đếm: người đang ở trong phòng

Ta dùng biến `cur` ("current", số người hiện tại), khởi tạo `0` (phòng trống). Gặp `'E'` thì `cur += 1`; gặp `'L'` thì `cur -= 1`. Biến này mô phỏng đúng đề: người vào làm tăng, người ra làm giảm.

In [3]:
cur = 0
for ch in "ELELEEL":
    if ch == "E":
        cur += 1
    else:
        cur -= 1
    print(ch, "-> cur =", cur)

E -> cur = 1
L -> cur = 0
E -> cur = 1
L -> cur = 0
E -> cur = 1
E -> cur = 2
L -> cur = 1


### 3.3. Ghi kỷ lục bằng `max`

Số ghế cần là **số người đông nhất từng có**. Ta giữ thêm biến `best` (kỷ lục) và sau mỗi bước cập nhật `best = max(best, cur)`. `max(a, b)` trả về số lớn hơn. **Kỷ lục không bị giảm** khi `cur` giảm, đó là điểm khác với `cur`.

In [4]:
print(max(3, 5), max(5, 3), max(4, 4))      # 5 5 4
best = 0
for cur in [1, 0, 1, 0, 1, 2, 1]:
    best = max(best, cur)
    print(f"cur = {cur}, best = {best}")

5 5 4
cur = 1, best = 1
cur = 0, best = 1
cur = 1, best = 1
cur = 0, best = 1
cur = 1, best = 1
cur = 2, best = 2
cur = 1, best = 2


### 3.4. `itertools.accumulate`: tổng cộng dồn

`accumulate([a, b, c])` cho dãy `a, a+b, a+b+c`. Nếu thay `E` bằng `+1` và `L` bằng `-1` thì tổng cộng dồn **chính là số người trong phòng sau mỗi giây**. Hàm `max(...)` trên dãy đó cho đáp án.

In [5]:
print(list(accumulate([1, 2, 3, 4])))                  # [1, 3, 6, 10]
steps = [1 if ch == "E" else -1 for ch in "ELELEEL"]
print(steps)                                           # [1, -1, 1, -1, 1, 1, -1]
print(list(accumulate(steps)))                         # [1, 0, 1, 0, 1, 2, 1]
print(max(accumulate(steps)))                          # 2

[1, 3, 6, 10]
[1, -1, 1, -1, 1, 1, -1]
[1, 0, 1, 0, 1, 2, 1]
2


### 3.5. `dict` ánh xạ ký tự → bước, và `list` làm ngăn xếp

- `delta = {"E": 1, "L": -1}`: tra `delta[ch]` cho bước tương ứng, gọn hơn `if/else`.
- `list.append(x)` thêm vào cuối; `list.pop()` lấy và bỏ phần tử cuối — dùng làm **kho ghế trống**: người vào lấy một ghế từ kho (nếu kho rỗng thì mua thêm ghế mới), người ra trả ghế về kho.

In [6]:
delta = {"E": 1, "L": -1}
print(delta["E"], delta["L"])           # 1 -1

free = []                               # kho ghế trống
free.append("ghế A")
free.append("ghế B")
print(free.pop(), "|", free)            # ghế B | ['ghế A']
print(bool([]), bool(["x"]))            # False True : list rỗng là "sai"

1 -1
ghế B | ['ghế A']
False True


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Với `s = "ELEELEELLL"`, ta ghi số người trong phòng **sau mỗi giây**:

| giây | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 |
|---|---|---|---|---|---|---|---|---|---|---|
| sự kiện | E | L | E | E | L | E | E | L | L | L |
| số người | 1 | 0 | 1 | 2 | 1 | 2 | 3 | 2 | 1 | 0 |

Ghế cần tại giây `t` bằng số người lúc đó. Để **mọi** giây đều đủ ghế ta cần **ít nhất bằng số người đông nhất**: `max = 3` (giây `6`). Với `3` ghế thì đủ, với `2` ghế thì giây `6` có `3` người — thiếu. Đáp án `3`.

Nhận xét quan trọng: **số người đông nhất** khác với **tổng số lần `E`** (ở đây `5` lần `E`, nhưng chỉ cần `3` ghế vì người ra trả ghế cho người sau) và khác với **số người lúc cuối** (`0`).

In [7]:
s = "ELEELEELLL"
cur, history = 0, []
for ch in s:
    cur += 1 if ch == "E" else -1
    history.append(cur)
print("số người sau mỗi giây:", history)
print("đông nhất:", max(history), "| tổng số E:", s.count("E"), "| lúc cuối:", history[-1])

số người sau mỗi giây: [1, 0, 1, 2, 1, 2, 3, 2, 1, 0]
đông nhất: 3 | tổng số E: 5 | lúc cuối: 0


## 5. Cách 1 — Brute Force / baseline: thử từng số ghế `c = 1, 2, 3, ...`

**Ý tưởng.** Đọc đề sát chữ: "số ghế **tối thiểu** sao cho đủ". Ta thử `c = 1, 2, 3, ...`; với mỗi `c` mô phỏng cả dãy và kiểm tra **có lúc nào số người vượt `c` không**. Số `c` nhỏ nhất đạt yêu cầu là đáp án. Đúng nhưng tốn công: mỗi lần thử là một lượt duyệt `s`.

In [8]:
def enough_chairs(s: str, c: int) -> bool:
    people = 0
    for ch in s:
        people += 1 if ch == "E" else -1
        if people > c:                       # có người không có ghế
            return False
    return True


def minimum_chairs_try_all(s: str) -> int:
    for c in range(1, len(s) + 1):           # nhiều nhất len(s) người cùng lúc
        if enough_chairs(s, c):
            return c
    return 0


for s, expected in examples:
    print(repr(s), "->", minimum_chairs_try_all(s), "(kỳ vọng", expected, ")")

'EEEEEEE' -> 7 (kỳ vọng 7 )
'ELELEEL' -> 2 (kỳ vọng 2 )
'ELEELEELLL' -> 3 (kỳ vọng 3 )


**Độ phức tạp (Cách 1).** Gọi `n = len(s)`. Tối đa `n` lần thử, mỗi lần duyệt `n` ký tự ⇒ **Time `O(n²)`**, **Space `O(1)`**.

**Nút thắt:** ta thử đi thử lại trong khi **một lượt duyệt đã đủ** để biết số người đông nhất — đáp án chính là con số đó, không cần "đoán rồi kiểm tra".

## 6. Ý tưởng thuật toán / bước đột phá

**Đáp án = số người đông nhất từng có.** Vì mỗi giây số ghế cần dùng bằng số người đang trong phòng, ghế tối thiểu chính là **kỷ lục** của số người. Chỉ cần **một lượt duyệt**:

1. `cur = 0`, `best = 0`.
2. Với mỗi ký tự: `E` ⇒ `cur += 1`; `L` ⇒ `cur -= 1`.
3. Sau mỗi bước: `best = max(best, cur)`.
4. Trả về `best`.

Lý do chỉ cần xét sau khi `cur` **tăng** là `cur` chỉ đạt đỉnh mới khi có người **vào**; nhưng cập nhật `max` sau **mọi** bước vẫn đúng và an toàn hơn.

In [9]:
s = "ELELEEL"
cur = best = 0
for i, ch in enumerate(s):
    cur += 1 if ch == "E" else -1
    best = max(best, cur)
    print(f"giây {i}: {ch} -> cur = {cur}, best = {best}")
print("đáp án:", best)

giây 0: E -> cur = 1, best = 1
giây 1: L -> cur = 0, best = 1
giây 2: E -> cur = 1, best = 1
giây 3: L -> cur = 0, best = 1
giây 4: E -> cur = 1, best = 1
giây 5: E -> cur = 2, best = 2
giây 6: L -> cur = 1, best = 2
đáp án: 2


## 7. Các cách tiếp theo

### Cách 2 — Một biến đếm `cur` và kỷ lục `best`

**Giải thích từng đoạn.**

1. `cur` là số người hiện tại, `best` là kỷ lục, đều bắt đầu `0`.
2. `for ch in s`: xử lý lần lượt từng giây.
3. `if ch == 'E': cur += 1` / `else: cur -= 1`.
4. `best = max(best, cur)`: ghi nhận kỷ lục sau mỗi giây.
5. Trả `best`.

In [10]:
def minimum_chairs_counter(s: str) -> int:
    cur = 0
    best = 0
    for ch in s:
        if ch == "E":
            cur += 1
        else:
            cur -= 1
        best = max(best, cur)
    return best


for s, expected in examples:
    print(repr(s), "->", minimum_chairs_counter(s), "(kỳ vọng", expected, ")")

'EEEEEEE' -> 7 (kỳ vọng 7 )
'ELELEEL' -> 2 (kỳ vọng 2 )
'ELEELEELLL' -> 3 (kỳ vọng 3 )


In [11]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionCounter:
    def minimumChairs(self, s: str) -> int:
        cur = best = 0
        for ch in s:
            cur += 1 if ch == "E" else -1
            best = max(best, cur)
        return best


print(SolutionCounter().minimumChairs("ELEELEELLL"))       # 3

3


**Độ phức tạp Cách 2:** **Time `O(n)`**, **Space `O(1)`**.

### Cách 3 — Tổng cộng dồn `accumulate` + `max`

**Giải thích.** Đổi `E → +1`, `L → -1` bằng `dict`, rồi `accumulate` ra dãy số người sau mỗi giây, cuối cùng `max`. Không có biến `cur`/`best` tường minh; đúng ý "tổng tiền tố lớn nhất". Vì `s` không rỗng và bước đầu của dãy hợp lệ luôn là `E`, `max` luôn `>= 1`; nếu dãy rỗng thì `max` lỗi, nên có thể thêm `default=0`.

In [12]:
def minimum_chairs_accumulate(s: str) -> int:
    delta = {"E": 1, "L": -1}
    return max(accumulate(delta[ch] for ch in s), default=0)


for s, expected in examples:
    print(repr(s), "->", minimum_chairs_accumulate(s), "(kỳ vọng", expected, ")")

'EEEEEEE' -> 7 (kỳ vọng 7 )
'ELELEEL' -> 2 (kỳ vọng 2 )
'ELEELEELLL' -> 3 (kỳ vọng 3 )


In [13]:
# Code nộp LeetCode cho Cách 3
class SolutionAccumulate:
    def minimumChairs(self, s: str) -> int:
        delta = {"E": 1, "L": -1}
        return max(accumulate(delta[ch] for ch in s), default=0)


print(SolutionAccumulate().minimumChairs("ELELEEL"))       # 2

2


**Độ phức tạp Cách 3:** **Time `O(n)`**, **Space `O(1)`** (generator + `accumulate` lười, không lưu cả dãy).

### Cách 4 — Mô phỏng thật: kho ghế trống

**Giải thích.** Thay vì đếm người, mô phỏng **ghế**: `free` là danh sách ghế đang trống, `chairs` là tổng số ghế đã phải có.

- `E`: nếu `free` còn ghế ⇒ `free.pop()` (lấy một ghế); nếu `free` rỗng ⇒ **mua thêm một ghế mới** (`chairs += 1`).
- `L`: trả một ghế về kho (`free.append(...)`).

Số ghế cần có = số ghế đã phải mua. Cách này đúng nghĩa "tối thiểu": ta chỉ mua ghế khi **không còn ghế trống nào**.

In [14]:
def minimum_chairs_simulate(s: str) -> int:
    free = []                    # các ghế đang trống (ghi số hiệu ghế)
    chairs = 0                   # tổng số ghế đã phải có
    for ch in s:
        if ch == "E":
            if free:
                free.pop()       # lấy một ghế trống
            else:
                chairs += 1      # không còn ghế trống: cần thêm một ghế mới
        else:
            free.append(chairs)  # người ra trả ghế (số hiệu không quan trọng)
    return chairs


for s, expected in examples:
    print(repr(s), "->", minimum_chairs_simulate(s), "(kỳ vọng", expected, ")")

'EEEEEEE' -> 7 (kỳ vọng 7 )
'ELELEEL' -> 2 (kỳ vọng 2 )
'ELEELEELLL' -> 3 (kỳ vọng 3 )


In [15]:
# Code nộp LeetCode cho Cách 4
class SolutionSimulate:
    def minimumChairs(self, s: str) -> int:
        free = 0                 # số ghế trống (chỉ cần đếm, không cần danh sách)
        chairs = 0               # tổng số ghế
        for ch in s:
            if ch == "E":
                if free:
                    free -= 1
                else:
                    chairs += 1
            else:
                free += 1
        return chairs


print(SolutionSimulate().minimumChairs("ELEELEELLL"))      # 3

3


**Độ phức tạp Cách 4:** **Time `O(n)`**, **Space `O(n)`** bản dùng `list` (hoặc `O(1)` bản chỉ đếm `free` như code nộp ở trên).

### Cách 5 — Công thức `E − L` trên mỗi tiền tố

**Giải thích.** Số người sau `i + 1` giây = (số `E` trong `s[:i+1]`) − (số `L` trong `s[:i+1]`). Lấy `max` trên mọi tiền tố. Cách này là **phát biểu toán học** của đề, tính trực tiếp bằng `str.count` trên lát cắt (không tối ưu: mỗi tiền tố đếm lại từ đầu).

In [16]:
def minimum_chairs_prefix(s: str) -> int:
    best = 0
    for i in range(1, len(s) + 1):
        prefix = s[:i]                                    # i giây đầu tiên
        people = prefix.count("E") - prefix.count("L")
        best = max(best, people)
    return best


for s, expected in examples:
    print(repr(s), "->", minimum_chairs_prefix(s), "(kỳ vọng", expected, ")")

'EEEEEEE' -> 7 (kỳ vọng 7 )
'ELELEEL' -> 2 (kỳ vọng 2 )
'ELEELEELLL' -> 3 (kỳ vọng 3 )


In [17]:
# Code nộp LeetCode cho Cách 5
class SolutionPrefix:
    def minimumChairs(self, s: str) -> int:
        return max(s[:i].count("E") - s[:i].count("L") for i in range(1, len(s) + 1))


print(SolutionPrefix().minimumChairs("ELELEEL"))           # 2

2


**Độ phức tạp Cách 5:** **Time `O(n²)`** (mỗi tiền tố đếm `O(n)`), **Space `O(n)`** (lát cắt). Chỉ để thấy công thức; với `n <= 50` vẫn tức thì.

## 8. TRACE từng bước

Ta trace Cách 2 (`cur` + `best`) trên `s = "ELEELEELLL"`, và Cách 4 (kho ghế trống) trên `s = "ELELEEL"`.

In [18]:
def trace_counter(s: str) -> int:
    print(f"s = {s!r}")
    cur = best = 0
    for i, ch in enumerate(s):
        before = cur
        cur += 1 if ch == "E" else -1
        best = max(best, cur)
        print(f"  giây {i}: {ch} ({'vào' if ch == 'E' else 'ra'}): cur {before} -> {cur}, best = {best}")
    return best


assert trace_counter("ELEELEELLL") == 3
assert trace_counter("EEEEEEE") == 7

s = 'ELEELEELLL'
  giây 0: E (vào): cur 0 -> 1, best = 1
  giây 1: L (ra): cur 1 -> 0, best = 1
  giây 2: E (vào): cur 0 -> 1, best = 1
  giây 3: E (vào): cur 1 -> 2, best = 2
  giây 4: L (ra): cur 2 -> 1, best = 2
  giây 5: E (vào): cur 1 -> 2, best = 2
  giây 6: E (vào): cur 2 -> 3, best = 3
  giây 7: L (ra): cur 3 -> 2, best = 3
  giây 8: L (ra): cur 2 -> 1, best = 3
  giây 9: L (ra): cur 1 -> 0, best = 3
s = 'EEEEEEE'
  giây 0: E (vào): cur 0 -> 1, best = 1
  giây 1: E (vào): cur 1 -> 2, best = 2
  giây 2: E (vào): cur 2 -> 3, best = 3
  giây 3: E (vào): cur 3 -> 4, best = 4
  giây 4: E (vào): cur 4 -> 5, best = 5
  giây 5: E (vào): cur 5 -> 6, best = 6
  giây 6: E (vào): cur 6 -> 7, best = 7


In [19]:
def trace_simulate(s: str) -> int:
    print(f"s = {s!r}")
    free = 0
    chairs = 0
    for i, ch in enumerate(s):
        if ch == "E":
            if free:
                free -= 1
                action = "lấy ghế trống"
            else:
                chairs += 1
                action = "mua thêm ghế"
        else:
            free += 1
            action = "trả ghế"
        print(f"  giây {i}: {ch} -> {action}; ghế trống = {free}, tổng ghế = {chairs}")
    return chairs


assert trace_simulate("ELELEEL") == 2

s = 'ELELEEL'
  giây 0: E -> mua thêm ghế; ghế trống = 0, tổng ghế = 1
  giây 1: L -> trả ghế; ghế trống = 1, tổng ghế = 1
  giây 2: E -> lấy ghế trống; ghế trống = 0, tổng ghế = 1
  giây 3: L -> trả ghế; ghế trống = 1, tổng ghế = 1
  giây 4: E -> lấy ghế trống; ghế trống = 0, tổng ghế = 1
  giây 5: E -> mua thêm ghế; ghế trống = 0, tổng ghế = 2
  giây 6: L -> trả ghế; ghế trống = 1, tổng ghế = 2


Bảng trace Cách 2 với `s = "ELEELEELLL"`:

| giây | ký tự | `cur` trước | `cur` sau | `best` sau |
|---|---|---|---|---|
| 0 | E | 0 | 1 | 1 |
| 1 | L | 1 | 0 | 1 |
| 2 | E | 0 | 1 | 1 |
| 3 | E | 1 | 2 | 2 |
| 4 | L | 2 | 1 | 2 |
| 5 | E | 1 | 2 | 2 |
| 6 | E | 2 | 3 | **3** |
| 7 | L | 3 | 2 | 3 |
| 8 | L | 2 | 1 | 3 |
| 9 | L | 1 | 0 | 3 |

Đáp án `3`. Bảng trace Cách 4 với `s = "ELELEEL"`: giây `0` `E` mua ghế #1 (tổng `1`); giây `1` `L` trả (trống `1`); giây `2` `E` lấy ghế trống (trống `0`); giây `3` `L` trả (trống `1`); giây `4` `E` lấy (trống `0`); giây `5` `E` **không còn ghế trống** ⇒ mua ghế #2 (tổng `2`); giây `6` `L` trả. Đáp án `2`.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (`cur` + `best`): `O(n)` thời gian, `O(1)` bộ nhớ, dễ đọc.

In [20]:
class Solution:
    def minimumChairs(self, s: str) -> int:
        cur = 0                                   # số người đang trong phòng
        best = 0                                  # số người đông nhất từng có = số ghế cần
        for ch in s:
            if ch == "E":
                cur += 1                          # người vào, cần thêm một ghế
            else:
                cur -= 1                          # người ra, trả một ghế
            best = max(best, cur)                 # ghi nhận kỷ lục
        return best


sol = Solution()
for s, expected in examples:
    print(repr(s), "->", sol.minimumChairs(s), "(kỳ vọng", expected, ")")

'EEEEEEE' -> 7 (kỳ vọng 7 )
'ELELEEL' -> 2 (kỳ vọng 2 )
'ELEELEELLL' -> 3 (kỳ vọng 3 )


## 10. Vì sao thuật toán đúng

- **Ghế cần tại mỗi giây = số người lúc đó.** Mỗi người đang trong phòng chiếm đúng một ghế; `cur` được cập nhật đúng theo `E`/`L` nên `cur` sau giây `i` đúng bằng số người trong phòng sau giây `i`.
- **Điều kiện đủ:** với `best` ghế, tại mọi giây số người `cur <= best` nên ai vào cũng có ghế.
- **Điều kiện cần (tối thiểu):** tại giây mà `cur = best`, có đúng `best` người trong phòng cùng lúc, mỗi người một ghế riêng ⇒ cần **ít nhất** `best` ghế. Nên `best` vừa đủ vừa nhỏ nhất.
- **Khởi tạo `best = 0`:** nếu có người vào (đề đảm bảo `s` không rỗng và dãy hợp lệ nên ký tự đầu là `E`), `best >= 1`. Giá trị `0` chỉ làm "mốc" ban đầu.
- **Mọi cách khác tương đương:** Cách 1 thử `c` tăng dần và dừng ở `c = max(cur)`; Cách 3 và 5 tính cùng dãy `cur`; Cách 4 mua ghế đúng lúc `cur` vượt tổng ghế hiện có, nên tổng ghế cuối cùng bằng kỷ lục của `cur`.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(s)` (`n <= 50`).

| Cách | Time | Space |
|---|---|---|
| 1. Thử từng số ghế `c` | `O(n²)` | `O(1)` |
| 2. `cur` + `best` | `O(n)` | `O(1)` |
| 3. `accumulate` + `max` | `O(n)` | `O(1)` |
| 4. Kho ghế trống | `O(n)` | `O(n)` (`O(1)` nếu chỉ đếm) |
| 5. `E − L` trên tiền tố | `O(n²)` | `O(n)` |

Không thể tốt hơn `O(n)`: phải đọc mọi ký tự để biết đỉnh.

## 12. Bộ test

Ta gom các cách vào `impls` và so với kỳ vọng.

In [21]:
sol = Solution()
impls = {
    "try_all": minimum_chairs_try_all,
    "counter": minimum_chairs_counter,
    "accumulate": minimum_chairs_accumulate,
    "simulate": minimum_chairs_simulate,
    "prefix": minimum_chairs_prefix,
    "leetcode": sol.minimumChairs,
}


def check_all(s, expected, label):
    for key, fn in impls.items():
        actual = fn(s)
        assert actual == expected, (label, key, s, actual, expected)
    print("PASS", label)


for idx, (s, expected) in enumerate(examples, 1):
    check_all(s, expected, f"ví dụ {idx}")

PASS ví dụ 1
PASS ví dụ 2
PASS ví dụ 3


### 12.1. Các trường hợp biên

In [22]:
check_all("E", 1, "một người vào")
check_all("EL", 1, "một người vào rồi ra")
check_all("ELELELEL", 1, "lần lượt từng người: chỉ cần 1 ghế")
check_all("EELL", 2, "hai người vào rồi hai người ra")
check_all("EEELLL", 3, "ba vào, ba ra")
check_all("EELELL", 2, "đỉnh 2")
check_all("EEL" + "EEL", 3, "đỉnh 2 rồi lên 3: E E L E E L -> 1,2,1,2,3,2")
check_all("E" * 50, 50, "50 người vào, không ai ra")
check_all("E" * 25 + "L" * 25, 25, "25 vào rồi 25 ra")
check_all("EL" * 25, 1, "50 ký tự xen kẽ")
check_all("EEEELLLLEE", 4, "đỉnh ở giữa, kết thúc bằng 2 người")
check_all("EEL" * 10, 11, "mỗi nhóm vào 2 ra 1: số người tăng dần 1 mỗi nhóm")

PASS một người vào
PASS một người vào rồi ra
PASS lần lượt từng người: chỉ cần 1 ghế
PASS hai người vào rồi hai người ra
PASS ba vào, ba ra
PASS đỉnh 2
PASS đỉnh 2 rồi lên 3: E E L E E L -> 1,2,1,2,3,2
PASS 50 người vào, không ai ra
PASS 25 vào rồi 25 ra
PASS 50 ký tự xen kẽ
PASS đỉnh ở giữa, kết thúc bằng 2 người
PASS mỗi nhóm vào 2 ra 1: số người tăng dần 1 mỗi nhóm


### 12.2. Phản ví dụ cho các cách làm sai

Hai lỗi phổ biến: (a) **đếm tổng số `'E'`** (người ra đã trả ghế nhưng ta vẫn tính); (b) **trả về số người lúc cuối** `cur` thay vì kỷ lục.

In [23]:
def wrong_count_enters(s):
    return s.count("E")                              # SAI: bỏ qua việc người ra trả ghế


def wrong_final_people(s):
    cur = 0
    for ch in s:
        cur += 1 if ch == "E" else -1
    return cur                                       # SAI: số người lúc cuối, không phải đỉnh


print(wrong_count_enters("ELELEEL"), "!=", minimum_chairs_counter("ELELEEL"))      # 4 so với 2
print(wrong_final_people("ELELEEL"), "!=", minimum_chairs_counter("ELELEEL"))      # 1 so với 2
assert wrong_count_enters("ELELEEL") != minimum_chairs_counter("ELELEEL")
assert wrong_final_people("ELELEEL") != minimum_chairs_counter("ELELEEL")

4 != 2
1 != 2


### 12.3. Đối chiếu ngẫu nhiên (có seed) và vét cạn

Đầu vào hợp lệ: độ dài `1..50`, chỉ `E`/`L`, và dãy **hợp lệ** (không có `L` khi phòng trống). Ta sinh bằng cách: ở mỗi bước, nếu phòng trống bắt buộc `E`, ngược lại chọn ngẫu nhiên `E` hoặc `L`. Đối chiếu với oracle viết khác (dùng `list` các số người rồi `max`). **Vét cạn** mọi dãy hợp lệ độ dài `1..14`.

In [24]:
def oracle(s):
    people = []
    cur = 0
    for ch in s:
        cur += 1 if ch == "E" else -1
        assert cur >= 0, "dãy không hợp lệ"
        people.append(cur)
    return max(people)


def is_valid(s):
    cur = 0
    for ch in s:
        cur += 1 if ch == "E" else -1
        if cur < 0:
            return False
    return 1 <= len(s) <= 50 and set(s) <= {"E", "L"}


rng = random.Random(3168)
for trial in range(4000):
    n = rng.randint(1, 50)
    p = rng.choice([0.3, 0.5, 0.7])                       # xác suất chọn E
    cur, chars = 0, []
    for _ in range(n):
        if cur == 0 or rng.random() < p:
            chars.append("E")
            cur += 1
        else:
            chars.append("L")
            cur -= 1
    s = "".join(chars)
    assert is_valid(s), s
    ref = oracle(s)
    for key, fn in impls.items():
        assert fn(s) == ref, (key, s)
print("PASS 4000 chuỗi hợp lệ ngẫu nhiên")

count = 0
for n in range(1, 15):
    for combo in itertools.product("EL", repeat=n):
        s = "".join(combo)
        if not is_valid(s):
            continue
        ref = oracle(s)
        for key, fn in impls.items():
            assert fn(s) == ref, (key, s)
        count += 1
print("PASS vét cạn", count, "chuỗi hợp lệ độ dài 1..14")

PASS 4000 chuỗi hợp lệ ngẫu nhiên


PASS vét cạn 7059 chuỗi hợp lệ độ dài 1..14


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Đếm tổng số `'E'`

In [25]:
def sai_dem_E(s):
    count = 0
    for ch in s:
        if ch == "E":
            count += 1
    return count


print("sai:", sai_dem_E("ELELEEL"), "| đúng:", Solution().minimumChairs("ELELEEL"))
print("sai:", sai_dem_E("ELEELEELLL"), "| đúng:", Solution().minimumChairs("ELEELEELLL"))
print("sai:", sai_dem_E("EEEEEEE"), "| đúng:", Solution().minimumChairs("EEEEEEE"), "(trùng vì không ai ra)")

sai: 4 | đúng: 2
sai: 5 | đúng: 3
sai: 7 | đúng: 7 (trùng vì không ai ra)


**Vì sao sai:** người ra trả ghế cho người sau, nên số ghế cần **ít hơn** tổng số người từng vào. Ví dụ 1 (không ai ra) che mất lỗi này; Ví dụ 2, 3 mới lộ. **Cách sửa:** theo dõi số người **đang** trong phòng (`cur`) và lấy kỷ lục.

### Lỗi 2: Trả về `cur` cuối cùng thay vì kỷ lục

In [26]:
def sai_tra_cur(s):
    cur = 0
    for ch in s:
        cur += 1 if ch == "E" else -1
    return cur


print("sai:", sai_tra_cur("ELEELEELLL"), "| đúng:", Solution().minimumChairs("ELEELEELLL"))
print("sai:", sai_tra_cur("ELELEEL"), "| đúng:", Solution().minimumChairs("ELELEEL"))

sai: 0 | đúng: 3
sai: 1 | đúng: 2


**Vì sao sai:** `cur` cuối cùng chỉ là số người **lúc kết thúc**; ghế phải đủ cho **thời điểm đông nhất** trước đó. **Cách sửa:** thêm `best = max(best, cur)` trong vòng lặp và trả `best`.

### Lỗi 3: Quên cập nhật `max` (hoặc đặt ra ngoài vòng lặp)

In [27]:
def sai_max_ngoai_vong(s):
    cur = best = 0
    for ch in s:
        cur += 1 if ch == "E" else -1
    best = max(best, cur)               # SAI: chỉ so sánh một lần, sau khi vòng lặp kết thúc
    return best


def sai_max_chi_khi_L(s):
    cur = best = 0
    for ch in s:
        if ch == "E":
            cur += 1
        else:
            cur -= 1
            best = max(best, cur)       # SAI: đỉnh xảy ra sau khi người VÀO, không phải sau khi người RA
    return best


print("sai:", sai_max_ngoai_vong("ELEELEELLL"), "| đúng:", Solution().minimumChairs("ELEELEELLL"))
print("sai:", sai_max_chi_khi_L("EEEEEEE"), "| đúng:", Solution().minimumChairs("EEEEEEE"))
print("sai:", sai_max_chi_khi_L("EEL"), "| đúng:", Solution().minimumChairs("EEL"))

sai: 0 | đúng: 3
sai: 0 | đúng: 7
sai: 1 | đúng: 2


**Vì sao sai:** kỷ lục phải được cập nhật **bên trong** vòng lặp, sau mỗi thay đổi của `cur` (hoặc ít nhất sau mỗi `E`, vì chỉ lúc đó `cur` tăng). Cập nhật ngoài vòng lặp chỉ thấy giá trị cuối; chỉ cập nhật khi gặp `L` thì bỏ qua đỉnh nếu chuỗi toàn `E`. **Cách sửa:** đặt `best = max(best, cur)` ngay trong thân vòng lặp.

### Lỗi 4: Nhầm hướng `+1/-1`

In [28]:
def sai_nham_huong(s):
    cur = best = 0
    for ch in s:
        cur += 1 if ch == "L" else -1   # SAI: L làm tăng, E làm giảm
        best = max(best, cur)
    return best


print("sai:", sai_nham_huong("EEEEEEE"), "| đúng:", Solution().minimumChairs("EEEEEEE"))
print("sai:", sai_nham_huong("ELELEEL"), "| đúng:", Solution().minimumChairs("ELELEEL"))

sai: 0 | đúng: 7
sai: 0 | đúng: 2


**Vì sao sai:** `E` (Enter) là **vào** ⇒ `+1`; `L` (Leave) là **ra** ⇒ `-1`. Đảo ngược làm `cur` âm và kỷ lục sai. **Cách sửa:** ghi chú bên cạnh code: `E` là `+1` (vào), `L` là `-1` (ra); kiểm tra bằng Ví dụ 1 (`EEEEEEE` phải ra `7`).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "**Số tối thiểu để lúc nào cũng đủ**" (ghế, phòng họp, đường ray, thu ngân...) ⇒ **giá trị lớn nhất của một đại lượng đang thay đổi** (cur) trên toàn dòng thời gian.
- Chuỗi sự kiện vào/ra ⇒ biến đếm `cur` (`+1` / `-1`) và **kỷ lục `best = max(best, cur)`** trong cùng vòng lặp.
- Đó cũng là bài **tổng tiền tố lớn nhất** (maximum prefix sum): `max(accumulate(...))`.
- Phân biệt ba con số: **tổng số lần vào**, **số người lúc cuối**, **số người đông nhất** — chỉ số cuối cùng mới là đáp án.
- Đọc kỹ ràng buộc "dãy hợp lệ": đảm bảo `cur` không bao giờ âm, nên không phải xử lý `L` khi phòng trống.
- Các bài cùng họ: *Meeting Rooms II (253)*, *Car Pooling (1094)*, *Maximum Population Year (1854)*, *Final Value of Variable After Performing Operations (2011)*.

## 15. Biến thể của bài

1. **Cho thời điểm vào/ra dạng khoảng** (`[start, end]`) và tìm số ghế/phòng tối thiểu (**Meeting Rooms II**) — sắp xếp các sự kiện rồi quét.
2. **Trả thêm giây đầu tiên đạt đỉnh** (hoặc mọi giây đạt đỉnh).
3. **Mỗi người ngồi lâu hơn (nhiều ghế/người)**: `E` chiếm `w` ghế.
4. **Dãy không đảm bảo hợp lệ:** phát hiện `L` khi phòng trống và báo lỗi/bỏ qua.
5. **Thời gian phòng đầy:** có bao nhiêu giây phòng đạt đúng số ghế tối thiểu?

In [29]:
def first_peak_second(s):
    cur = best = 0
    first = -1
    for i, ch in enumerate(s):
        cur += 1 if ch == "E" else -1
        if cur > best:
            best, first = cur, i
    return first, best


def minimum_chairs_weighted(s, w):
    """Mỗi người chiếm w ghế."""
    return w * Solution().minimumChairs(s)


def minimum_chairs_safe(s):
    """Bỏ qua L khi phòng trống (dãy không hợp lệ)."""
    cur = best = 0
    for ch in s:
        if ch == "E":
            cur += 1
        elif cur > 0:
            cur -= 1
        best = max(best, cur)
    return best


def seconds_at_peak(s):
    cur = best = 0
    history = []
    for ch in s:
        cur += 1 if ch == "E" else -1
        history.append(cur)
    best = max(history)
    return sum(1 for v in history if v == best)


print(first_peak_second("ELEELEELLL"))        # (6, 3)
print(minimum_chairs_weighted("ELELEEL", 4))  # 8
print(minimum_chairs_safe("LLEE"))            # 2 (hai L đầu bị bỏ qua)
print(seconds_at_peak("ELELEEL"))             # 1 : chỉ giây 5 có 2 người
assert first_peak_second("ELEELEELLL") == (6, 3)
assert minimum_chairs_weighted("ELELEEL", 4) == 8
assert minimum_chairs_safe("LLEE") == 2
assert seconds_at_peak("ELELEEL") == 1
assert seconds_at_peak("EEL" + "EEL") == 1

(6, 3)
8
2
1


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử từng số ghế `c` | mô phỏng kiểm tra, lấy `c` nhỏ nhất đủ | `O(n²)` | `O(1)` | hiểu đề "tối thiểu"; không nên nộp |
| 2. `cur` + `best` | một biến đếm, ghi kỷ lục | `O(n)` | `O(1)` | **nên nộp**: ngắn, nhanh, rõ |
| 3. `accumulate` + `max` | tổng cộng dồn `+1/-1`, lấy lớn nhất | `O(n)` | `O(1)` | thích viết ngắn, quen "tổng tiền tố" |
| 4. Kho ghế trống | mua ghế khi không còn ghế trống | `O(n)` | `O(1)`-`O(n)` | mô phỏng sát đề, hợp biến thể có ghế riêng |
| 5. `E − L` trên tiền tố | `count` trên từng lát cắt | `O(n²)` | `O(n)` | chỉ để thấy công thức |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `max(2, 7)`, `"EEL".count("E")`, `list(accumulate([1, 1, -1, 1]))`, `Solution().minimumChairs("EEL")`, `Solution().minimumChairs("ELEL")`.

**Bài 2 (tính tay).** Với `s = "EEELELLEEL"`: lập bảng số người sau mỗi giây và cho biết số ghế tối thiểu và giây đầu tiên đạt đỉnh.

**Bài 3 (điền TODO).** Hoàn thành `minimum_chairs_todo(s)` theo Cách 2.

In [30]:
def minimum_chairs_todo(s):
    # TODO 1: cur = 0, best = 0
    # TODO 2: với mỗi ch trong s: E thì cur += 1, L thì cur -= 1; rồi best = max(best, cur)
    # TODO 3: trả về best
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `minimum_chairs_by_counts(s)` dùng **hai biến đếm** `entered` và `left` (số người đã vào, số người đã ra) thay cho `cur`: số người trong phòng là `entered - left`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `people_timeline(s)` trả về danh sách số người sau mỗi giây. (b) Viết `car_pooling_peak(events)` với `events` là danh sách `(giây, +k hoặc -k)` (nhiều người cùng lúc) đã sắp theo giây, trả về số người đông nhất.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `7`; `"EEL".count("E") = 2`; `[1, 2, 1, 2]`; `"EEL"` có số người `1, 2, 1` nên `2`; `"ELEL"` có `1, 0, 1, 0` nên `1`. Bài 3: đặt `best = max(best, cur)` **trong** vòng lặp.
- **Mức 2 (vừa):** Bài 2: số người `1, 2, 3, 2, 3, 2, 1, 2, 3, 2` ⇒ đỉnh `3`, giây đầu tiên đạt đỉnh là giây `2`. Bài 4: `entered += 1` khi `E`, `left += 1` khi `L`, `best = max(best, entered - left)`.
- **Mức 3 (sâu):** Bài 5(a): thêm `history.append(cur)` mỗi bước. 5(b): với mỗi `(giây, delta)`: `cur += delta`, `best = max(best, cur)`; nếu cùng một giây có cả vào lẫn ra thì xử lý **ra trước** để không đếm dư (chỉ cần nhắc trong ghi chú).

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [31]:
# Bài 1
assert max(2, 7) == 7
assert "EEL".count("E") == 2
assert list(accumulate([1, 1, -1, 1])) == [1, 2, 1, 2]
assert Solution().minimumChairs("EEL") == 2
assert Solution().minimumChairs("ELEL") == 1

# Bài 2
def people_timeline(s):
    cur, history = 0, []
    for ch in s:
        cur += 1 if ch == "E" else -1
        history.append(cur)
    return history


timeline = people_timeline("EEELELLEEL")
assert timeline == [1, 2, 3, 2, 3, 2, 1, 2, 3, 2]
assert max(timeline) == 3 == Solution().minimumChairs("EEELELLEEL")
assert timeline.index(max(timeline)) == 2


# Bài 3
def minimum_chairs_todo_ans(s):
    cur = best = 0
    for ch in s:
        if ch == "E":
            cur += 1
        else:
            cur -= 1
        best = max(best, cur)
    return best


for s, expected in examples:
    assert minimum_chairs_todo_ans(s) == expected


# Bài 4
def minimum_chairs_by_counts(s):
    entered = left = best = 0
    for ch in s:
        if ch == "E":
            entered += 1
        else:
            left += 1
        best = max(best, entered - left)
    return best


for s, expected in examples:
    assert minimum_chairs_by_counts(s) == expected


# Bài 5(a) đã viết ở trên: people_timeline
assert people_timeline("ELELEEL") == [1, 0, 1, 0, 1, 2, 1]


# Bài 5(b)
def car_pooling_peak(events):
    cur = best = 0
    for _, delta in events:
        cur += delta
        best = max(best, cur)
    return best


assert car_pooling_peak([(1, 2), (2, 3), (3, -2), (4, 4), (5, -5)]) == 7
assert car_pooling_peak([(1, 1), (2, -1), (3, 1)]) == 1
rng2 = random.Random(5)
for _ in range(300):
    n = rng2.randint(1, 50)
    cur, chars = 0, []
    for _ in range(n):
        if cur == 0 or rng2.random() < 0.5:
            chars.append("E"); cur += 1
        else:
            chars.append("L"); cur -= 1
    s = "".join(chars)
    assert minimum_chairs_by_counts(s) == Solution().minimumChairs(s) == max(people_timeline(s))
print("PASS toàn bộ đáp án")

PASS toàn bộ đáp án


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (đỉnh của số người đang có mặt)

```python
class Solution:
    def minimumChairs(self, s: str) -> int:
        cur = best = 0
        for ch in s:
            cur += 1 if ch == "E" else -1
            best = max(best, cur)
        return best
```

**Checklist**

- [ ] `E` là `+1` (vào), `L` là `-1` (ra)?
- [ ] Có biến `best` ghi **kỷ lục** và cập nhật **trong** vòng lặp (không chỉ trả `cur` cuối)?
- [ ] Không trả tổng số `'E'` (người ra đã trả ghế)?
- [ ] Đã thử `"EEEEEEE"` (`7`), `"ELELEEL"` (`2`), `"ELEELEELLL"` (`3`), và `"E"` (`1`)?
- [ ] Hiểu vì sao đáp án là **số người đông nhất cùng lúc** (đủ và tối thiểu)?
- [ ] Skeleton đúng `class Solution` / `def minimumChairs(self, s: str) -> int`?